# Joinville — Consolidação dos dados meteorológicos
**5-min · horário · diário — de todos os arquivos brutos para um dataset limpo por estação.**

Notebook reprodutível (VS Code ou Google Colab). Ele lê o arquivo bruto das estações
automáticas (Defesa Civil de Joinville), consolida cada estação em **três resoluções**
temporais, e gera um **inventário de disponibilidade por variável** para cada estação.

Regras (rigor científico): nada é interpolado ou inventado — apenas produtos nativos do
datalogger (5-min/horário instantâneos+médias; diário extremos+totais). Vento em m/s,
horário local (America/São_Paulo), sentinelas do Campbell (-100/NAN) removidas, QC por
faixas físicas. Lacunas reais permanecem lacunas.

**Como usar:** edite `PROJECT` na célula de configuração para apontar para a pasta do
projeto (a que contém `meteo/`). Deixe `REBUILD=False` para apenas carregar/plotar os
datasets já prontos em `datasets/`; use `REBUILD=True` para reconstruir tudo do zero.

In [ ]:
# Colab / ambiente novo: descomente para instalar as dependências
# !pip -q install pandas numpy matplotlib python-calamine pyarrow openpyxl

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# >>> EDITE AQUI: pasta do projeto (contém meteo/ com defesa_civil/, meteo_add/, PLUVIOMETROS/, *_raw.csv)
PROJECT = Path(r"C:/Users/air_p/Desktop/2026/PostDoc_UDESC/projeto_resposta_eventos")
# Google Colab (Drive):
#   from google.colab import drive; drive.mount("/content/drive")
#   PROJECT = Path("/content/drive/MyDrive/.../projeto_resposta_eventos")

METEO = PROJECT / "meteo"
OUT   = PROJECT / "datasets"            # saídas: 5min/ hourly/ daily/
FIGS  = PROJECT / "datasets" / "figs"
TZ    = "America/Sao_Paulo"
REBUILD = False                         # True = reconstruir do bruto; False = só carregar/plotar
for p in (OUT/"5min", OUT/"hourly", OUT/"daily", FIGS): p.mkdir(parents=True, exist_ok=True)
print("PROJECT:", PROJECT, "| existe:", PROJECT.exists())

## 1 · Leitores TOA5
Um único leitor mapeia os nomes de campo do Campbell para um esquema padrão, converte
vento km/h→m/s, remove sentinelas e aplica QC. Serve para `.dat` e para as planilhas
`*_HR/_DIARIA` (uma aba por ano).

In [ ]:
MAP_INST = {"TIMESTAMP":"date","Temp_Ar_Avg":"temp","Umid_Rel":"umid","R_Solar_Avg":"solar",
    "V_Vento_5":"ws","V_Vento_Horaria":"ws","D_Vento_5":"wd","D_Vento_Horaria":"wd",
    "Rajada":"gust","Dir_Rajada":"gust_dir","Chuva_Tot":"prec","PressATM":"pressure",
    "Orvalho":"dewpoint","Ind_Calor":"heat_index","WindChill":"wind_chill",
    "Nivel":"level","Nivel_Max":"level_max","Nivel_Min":"level_min"}
MAP_DAILY = {"TIMESTAMP":"date","Temp_Ar_Max":"temp_max","Temp_Ar_Min":"temp_min",
    "Umid_Rel_Max":"umid_max","Umid_Rel_Min":"umid_min","Rad_Total_Tot":"solar_total",
    "Rajada":"gust_max","Dir_Rajada":"gust_dir","Chuva_Tot":"prec","Nivel_Max":"level_max",
    "Nivel_Min":"level_min","Orvalho":"dewpoint","Ind_Calor":"heat_index","WindChill":"wind_chill"}
NA_SENT = ["NAN","-100","-100.0","-6999","-6999.0","-7999","-99999","-144.9"]
WIND = {"ws","gust","gust_max"}
QC = {"temp":(-10,50),"temp_max":(-10,50),"temp_min":(-10,50),"umid":(0,100),"umid_max":(0,100),
    "umid_min":(0,100),"prec":(0,500),"ws":(0,100),"gust":(0,120),"gust_max":(0,120),"wd":(0,360),
    "gust_dir":(0,360),"solar":(0,1600),"solar_total":(0,45000),"pressure":(800,1050),
    "dewpoint":(-10,40),"level":(-50,5000),"level_max":(-50,5000),"level_min":(-50,5000)}
MIN_DATE = pd.Timestamp("2010-01-01")

def _finalize(df, mapping):
    keep={c:mapping[c] for c in df.columns if c in mapping}
    df=df[list(keep)].rename(columns=keep); df=df.loc[:,~df.columns.duplicated()]
    d=df["date"]
    if not pd.api.types.is_datetime64_any_dtype(d):
        p=pd.to_datetime(d,format="%Y-%m-%d %H:%M:%S",errors="coerce")
        miss=p.isna()&d.notna()
        if miss.any(): p.loc[miss]=pd.to_datetime(d[miss],errors="coerce")
        d=p
    df=df.assign(date=d).dropna(subset=["date"]); df=df[df["date"]>=MIN_DATE]
    for c in [c for c in df.columns if c!="date"]: df[c]=pd.to_numeric(df[c],errors="coerce")
    for c in WIND & set(df.columns): df[c]=df[c]/3.6
    for c,(lo,hi) in QC.items():
        if c in df.columns: df.loc[(df[c]<lo)|(df[c]>hi),c]=np.nan
    return df.sort_values("date").reset_index(drop=True)

def parse_dat(path, mapping=MAP_INST):
    path=Path(path)
    names=pd.read_csv(path,skiprows=1,nrows=1,header=None,encoding="latin-1").iloc[0].tolist()
    df=pd.read_csv(path,skiprows=4,header=None,names=names,encoding="latin-1",na_values=NA_SENT,low_memory=False)
    return _finalize(df,mapping)

def parse_xlsx(path, mapping=MAP_INST):
    path=Path(path)
    try: xl=pd.ExcelFile(path,engine="calamine"); eng="calamine"
    except Exception: xl=pd.ExcelFile(path); eng=None
    frames=[]
    for sh in xl.sheet_names:
        raw=pd.read_excel(path,sheet_name=sh,header=None,engine=eng)
        hdr=next((i for i in range(min(8,len(raw))) if (raw.iloc[i].astype(str).str.strip()=="TIMESTAMP").any()),None)
        if hdr is None: continue
        b=raw.iloc[hdr+1:].copy(); b.columns=raw.iloc[hdr].tolist()
        b=b.loc[:,~b.columns.duplicated()].replace(NA_SENT,np.nan); frames.append(b)
    if not frames: return pd.DataFrame(columns=list(mapping.values()))
    return _finalize(pd.concat(frames,ignore_index=True),mapping)

def load_raw_csv(path):  # legacy *_raw.csv (já em m/s, UTC)
    df=pd.read_csv(path); df["date"]=pd.to_datetime(df["date"],utc=True,errors="coerce").dt.tz_convert(TZ).dt.tz_localize(None)
    return df.dropna(subset=["date"])
print("leitores TOA5 prontos")

## 2 · Manifesto das estações
Onde vive cada fonte por estação: séries legadas `*_raw.csv`, dumps de 5-min em
`meteo_add/`, tabelas atuais em `defesa_civil/`, os **fragmentos** do histórico em
`HISTORICO/Joinville/<pasta>/` (essenciais — preenchem 2021–2026 no 5-min), e as
planilhas horárias/diárias `*_HR/_DIARIA`.

In [ ]:
HIST = METEO/"defesa_civil"/"HISTORICO"/"Joinville"
DEF  = METEO/"defesa_civil"
ADD  = METEO/"meteo_add"

# code: dict(raw=[...], dat=[...], histfolder=..., hr_xlsx=.., di_xlsx=.., hr_dat=.., di_dat=..)
STATIONS = {
 "ceasa":       dict(raw=[METEO/"ceasa_raw.csv"], dat=[ADD/"CEASA_CEASA_5.dat", DEF/"CEASA_5.dat"],
                     hist="Ceasa-Met-PB10", hr_xlsx="CEASA_HR (2011-2026).xlsx", di_xlsx="CEASA_DIARIA (2011-2026).xlsx",
                     hr_dat="CEASA_HR.dat", di_dat="CEASA_DIARIA.dat"),
 "iateclube":   dict(raw=[METEO/"iateclube_raw.csv"], dat=[ADD/"IATECLUBE_IATCLUB_5.dat", DEF/"IATCLUB_5.dat"],
                     hist="IatClub-HidroMet-PB7", hr_xlsx="IATCLUB_HR (2011-2026).xlsx", di_xlsx="IATCLUB_DIARIA (2011-2026).xlsx",
                     hr_dat="IATCLUB_HR.dat", di_dat="IATCLUB_DIARIA.dat"),
 "flotflux":    dict(raw=[METEO/"flotflux_raw.csv"], dat=[ADD/"FLOT_FLUX_FLOTFLUX_5.dat", DEF/"FLOTFLUX_5.dat"],
                     hist="flot-flux-HidroMet-PB3", hr_xlsx="FLOTFLUX_HR (2011-2026).xlsx", di_xlsx="FLOTFLUX_DIARIA (2011-2026).xlsx",
                     hr_dat="FLOTFLUX_HR.dat", di_dat="FLOTFLUX_DIARIA.dat"),
 "aguasdejoi":  dict(raw=[METEO/"aguasdejoi_raw.csv"], dat=[ADD/"AGUASDEJOINVILLE_AGUAS_5.dat"],
                     hist="Aguas-HidroMet-PB11", hr_xlsx="AGUAS_HR (2011-2024).xlsx", di_xlsx="AGUAS_DIARIA (2011-2024).xlsx"),
 "cubatao":     dict(raw=[METEO/"cubatao_raw.csv"], dat=[], hist="Cubatao-HidroMet-PB9",
                     hr_xlsx="CUBATAO_HR (2011-2024).xlsx", di_xlsx="CUBATAO_DIARIA (2011-2024).xlsx"),
 "itaum":       dict(raw=[METEO/"itaum_raw.csv"], dat=[], hist="ITAUM-Caixa-dagua-Met-PB15",
                     hr_xlsx="ITAUM_HR (2011-2024).xlsx", di_xlsx="ITAUM_DIARIA (2011-2024).xlsx"),
 "rodovia":     dict(raw=[METEO/"rodovia_raw.csv"], dat=[], hist="Estr_Sul-Met-PB13",
                     hr_xlsx="ESTR_SUL_HR (2011-2022).xlsx", di_xlsx="ESTR_SUL_DIARIA (2011-2022).xlsx"),
 "divobras":    dict(raw=[], dat=[DEF/"DIVOBRAS_5.dat"], hist="DIVOBRAS-Hidro-PB",
                     hr_xlsx="DIVOBRAS_HR (2011-2024).xlsx", di_xlsx="DIVOBRAS_DIARIA (2011-2024).xlsx",
                     hr_dat="DIVOBRAS_HR.dat", di_dat="DIVOBRAS_DIARIA.dat"),
 "guanabara":   dict(raw=[], dat=[ADD/"GUANABARA_GUANABARA_5.dat", ADD/"GUANABARA_GUANABARA_HIDRO_5.dat"],
                     hist="Guanabara-HidroMet-PB5", hr_xlsx="GUANABARA_HR (2011-2024).xlsx", di_xlsx="GUANABARA_DIARIA (2011-2024).xlsx"),
 "jardimparaiso":dict(raw=[], dat=[ADD/"PARAISO_J_PARAISO_5.dat", ADD/"PARAISO_PARAISO_HIDRO_5.dat", DEF/"J_PARAISO_5.dat"],
                     hist="Paraiso-Hidro-PB8", hr_xlsx="J_PARAISO_HR (2011-2026).xlsx", di_xlsx="J_PARAISO_DIARIA (2011-2026).xlsx",
                     hr_dat="J_PARAISO_HR.dat", di_dat="J_PARAISO_DIARIA.dat"),
}
STD5 = ["date","temp","umid","prec","ws","wd","gust","gust_dir","solar","pressure",
        "dewpoint","heat_index","wind_chill","level","level_max","level_min"]
print(len(STATIONS), "estações no manifesto")

## 3 · Construir as séries de 5 minutos
Junta séries legadas + dumps + tabela atual + **fragmentos do histórico**, deduplica por
timestamp mantendo o registro mais completo. Os fragmentos (`<TABELA>_5_<data>-NNNN.dat`)
são concatenados e lidos de uma vez (as linhas de cabeçalho viram NaT e são descartadas).

In [ ]:
import tempfile
def read_fragment_folder(folder):
    """Concatena todos os fragmentos *_5*.dat de uma pasta do histórico e parseia."""
    folder=Path(folder)
    frags=sorted(folder.glob("*_5_[0-9]*.dat"))+sorted(folder.glob("*_5.dat"))
    if not frags: return pd.DataFrame(columns=STD5)
    with tempfile.NamedTemporaryFile("w",suffix=".dat",delete=False,encoding="latin-1") as tf:
        for fp in frags:
            try: tf.write(fp.read_text(encoding="latin-1"))
            except Exception: pass
        tmp=tf.name
    try: return parse_dat(tmp, MAP_INST)
    finally: Path(tmp).unlink(missing_ok=True)

def dedup(df, cols):
    for c in cols:
        if c not in df: df[c]=np.nan
    df=df[cols].copy(); df["date"]=pd.to_datetime(df["date"])
    df["_n"]=df.drop(columns=["date"]).notna().sum(axis=1)
    return (df.sort_values(["date","_n"],ascending=[True,False]).drop_duplicates("date",keep="first")
              .drop(columns="_n").reset_index(drop=True))

def build_5min(code, cfg):
    frames=[load_raw_csv(p) for p in cfg["raw"] if Path(p).exists()]
    frames+=[parse_dat(p,MAP_INST) for p in cfg["dat"] if Path(p).exists()]
    if cfg.get("hist") and (HIST/cfg["hist"]).exists():
        frames.append(read_fragment_folder(HIST/cfg["hist"]))
    if not frames: return pd.DataFrame(columns=STD5)
    m=dedup(pd.concat(frames,ignore_index=True), STD5)
    return m[["date"]+[c for c in STD5 if c!="date" and m[c].notna().any()]]

if REBUILD:
    for code,cfg in STATIONS.items():
        m=build_5min(code,cfg); m.to_csv(OUT/"5min"/f"{code}.csv",index=False)
        print(f"{code:13s} {len(m):>9,} linhas | {str(m.date.min())[:10]}..{str(m.date.max())[:10]}")
else:
    print("REBUILD=False — usando datasets/5min já prontos")

## 4 · Construir as séries horárias e diárias
**Horário** = planilhas nativas `*_HR` + `.dat` recente. **Diário** = extremos/totais
nativos de `*_DIARIA` + **médias derivadas** (média das horas do dia; `n_hours` = nº de
horas usadas).

In [ ]:
def build_hourly(code,cfg):
    fr=[]
    if cfg.get("hr_xlsx") and (ADD/cfg["hr_xlsx"]).exists(): fr.append(parse_xlsx(ADD/cfg["hr_xlsx"],MAP_INST))
    if cfg.get("hr_dat") and (DEF/cfg["hr_dat"]).exists():   fr.append(parse_dat(DEF/cfg["hr_dat"],MAP_INST))
    if not fr: return pd.DataFrame(columns=STD5)
    h=pd.concat(fr,ignore_index=True); h["date"]=pd.to_datetime(h["date"]).dt.floor("h")
    h=dedup(h,STD5); return h[["date"]+[c for c in STD5 if c!="date" and h[c].notna().any()]]

def build_daily(code,cfg,hourly):
    DCOLS=["date","temp_max","temp_min","umid_max","umid_min","solar_total","gust_max","gust_dir",
           "prec","level_max","level_min","dewpoint","heat_index","wind_chill"]
    fr=[]
    if cfg.get("di_xlsx") and (ADD/cfg["di_xlsx"]).exists(): fr.append(parse_xlsx(ADD/cfg["di_xlsx"],MAP_DAILY))
    if cfg.get("di_dat") and (DEF/cfg["di_dat"]).exists():   fr.append(parse_dat(DEF/cfg["di_dat"],MAP_DAILY))
    d=dedup(pd.concat(fr,ignore_index=True),DCOLS) if fr else pd.DataFrame(columns=DCOLS)
    d["date"]=pd.to_datetime(d["date"]).dt.floor("D")
    # médias derivadas do horário (documentado)
    if len(hourly):
        hh=hourly.copy(); hh["date"]=pd.to_datetime(hh["date"]).dt.floor("D")
        agg={f"{s}_mean":(s,"mean") for s in ("temp","umid","ws","pressure","solar") if s in hh}
        means=hh.groupby("date").agg(n_hours=("date","size"),**agg).reset_index()
        d=d.merge(means,on="date",how="outer").sort_values("date")
    return d[["date"]+[c for c in d.columns if c!="date" and d[c].notna().any()]]

if REBUILD:
    for code,cfg in STATIONS.items():
        h=build_hourly(code,cfg); h.to_csv(OUT/"hourly"/f"{code}.csv",index=False)
        dd=build_daily(code,cfg,h); dd.to_csv(OUT/"daily"/f"{code}.csv",index=False)
        print(f"{code:13s} horário {len(h):>7,} | diário {len(dd):>5,}")
else:
    print("REBUILD=False — usando datasets/hourly e datasets/daily já prontos")

## 5 · Inventário de cobertura (heatmap por estação × mês)
Completude mensal = observações presentes ÷ esperadas. Cinza = mês sem dado.

In [ ]:
PER_DAY={"5min":288,"hourly":24,"daily":1}
NAMES={"ceasa":"Ceasa","iateclube":"Iate Clube","flotflux":"Cachoeira (Flotflux)","cubatao":"Cubatão",
"itaum":"Itaum","aguasdejoi":"Águas (Bucarein)","rodovia":"Rodovia do Arroz","guanabara":"Guanabara",
"jardimparaiso":"Paraíso","divobras":"Unidade de Obras"}
ORDER=list(NAMES)
BLUES=LinearSegmentedColormap.from_list("b",["#eaf2fc","#cde2fb","#86b6ef","#3987e5","#1c5cab","#0d366b"]); BLUES.set_bad("#efeeea")

def coverage_heatmap(tier):
    per=PER_DAY[tier]; rows=[]
    for code in ORDER:
        fp=OUT/tier/f"{code}.csv"
        if not fp.exists(): continue
        d=pd.read_csv(fp,usecols=["date"]); d["date"]=pd.to_datetime(d["date"],errors="coerce")
        d=d.dropna(subset=["date"]); d=d[d["date"]>=MIN_DATE]
        for p,n in d.groupby(d["date"].dt.to_period("M")).size().items(): rows.append((code,p,n))
    c=pd.DataFrame(rows,columns=["station","mp","n"]); c["days"]=c["mp"].dt.days_in_month
    c["pct"]=(100*c["n"]/(c["days"]*per)).clip(0,100)
    full=pd.period_range("2011-01","2026-07",freq="M")
    piv=c.pivot_table(index="station",columns="mp",values="pct").reindex(index=[o for o in ORDER if o in set(c.station)],columns=full)
    fig,ax=plt.subplots(figsize=(15,0.5*len(piv)+1.2))
    im=ax.imshow(np.ma.masked_invalid(piv.values),aspect="auto",cmap=BLUES,vmin=0,vmax=100,interpolation="nearest")
    ax.set_yticks(range(len(piv))); ax.set_yticklabels([NAMES[c] for c in piv.index],fontsize=11)
    yr=[i for i,p in enumerate(full) if p.month==1]; ax.set_xticks(yr); ax.set_xticklabels([full[i].year for i in yr])
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_xticks(np.arange(-.5,len(full)),minor=True); ax.set_yticks(np.arange(-.5,len(piv)),minor=True)
    ax.grid(which="minor",color="white",lw=.6); ax.tick_params(which="both",length=0)
    cb=fig.colorbar(im,ax=ax,fraction=0.025,pad=0.01); cb.set_label("Cobertura mensal (%)"); cb.outline.set_visible(False)
    ax.set_title(f"Cobertura — série {tier}",fontsize=13,fontweight="600",loc="left",pad=10)
    plt.tight_layout(); plt.savefig(FIGS/f"coverage_{tier}.png",dpi=140,bbox_inches="tight",facecolor="white"); plt.show()

for t in ("5min","hourly","daily"): coverage_heatmap(t)

## 6 · Inventário por VARIÁVEL (para cada estação)
Para cada estação, disponibilidade mensal de **cada variável** — mostra, por exemplo,
quando um sensor específico caiu (ex.: temperatura da Ceasa) enquanto os demais seguiram.

In [ ]:
def variable_inventory(code, tier="hourly"):
    fp=OUT/tier/f"{code}.csv"
    if not fp.exists(): print("sem",fp); return
    df=pd.read_csv(fp); df["date"]=pd.to_datetime(df["date"],errors="coerce")
    df=df.dropna(subset=["date"]); df=df[df["date"]>=MIN_DATE]
    vars=[c for c in df.columns if c!="date"]; df["mp"]=df["date"].dt.to_period("M")
    per=PER_DAY[tier]
    full=pd.period_range(df["mp"].min(),df["mp"].max(),freq="M")
    exp=pd.Series({p:p.days_in_month*per for p in full})
    M=pd.DataFrame(index=vars,columns=full,dtype=float)
    g=df.groupby("mp")
    for v in vars:
        cnt=g[v].apply(lambda s:s.notna().sum())
        M.loc[v]=(100*cnt/exp).clip(0,100)
    fig,ax=plt.subplots(figsize=(14,0.42*len(vars)+1))
    im=ax.imshow(np.ma.masked_invalid(M.values.astype(float)),aspect="auto",cmap=BLUES,vmin=0,vmax=100,interpolation="nearest")
    ax.set_yticks(range(len(vars))); ax.set_yticklabels(vars,fontsize=10)
    yr=[i for i,p in enumerate(full) if p.month==1]; ax.set_xticks(yr); ax.set_xticklabels([full[i].year for i in yr])
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_xticks(np.arange(-.5,len(full)),minor=True); ax.set_yticks(np.arange(-.5,len(vars)),minor=True)
    ax.grid(which="minor",color="white",lw=.5); ax.tick_params(which="both",length=0)
    cb=fig.colorbar(im,ax=ax,fraction=0.025,pad=0.01); cb.set_label("Disponibilidade mensal (%)"); cb.outline.set_visible(False)
    ax.set_title(f"{NAMES.get(code,code)} — disponibilidade por variável ({tier})",fontsize=13,fontweight="600",loc="left",pad=10)
    plt.tight_layout(); plt.savefig(FIGS/f"inventory_{code}_{tier}.png",dpi=140,bbox_inches="tight",facecolor="white"); plt.show()

for code in ORDER: variable_inventory(code, tier="hourly")

## 7 · Exemplo — série temporal de uma variável
Ajuste `code`, `tier` e `var` para inspecionar qualquer variável.

In [ ]:
code, tier, var = "iateclube", "daily", "temp_mean"
df=pd.read_csv(OUT/tier/f"{code}.csv"); df["date"]=pd.to_datetime(df["date"])
fig,ax=plt.subplots(figsize=(14,3.2))
ax.plot(df["date"],df[var],lw=0.7,color="#2a78d6")
ax.set_title(f"{NAMES.get(code,code)} — {var} ({tier})",loc="left",fontweight="600")
for s in ("top","right"): ax.spines[s].set_visible(False)
ax.grid(axis="y",color="#e1e0d9"); plt.tight_layout(); plt.show()